# Instituto Superior de Engenharia do Porto (ISEP)
## Departamento de Engenharia Electrotécnica (DEE)
### Unidade Curricular: Sinais e Sistemas (SINSI)

---
# PARTE 1: Correlação e Autocorrelação de Sinais

**Tópicos Abordados:**
- Fundamentos teóricos da Correlação e Autocorrelação (Tempo Contínuo e Discreto).
- Medição de semelhança e coerência temporal (*Slide & Integrate* sem inversão temporal).
- Demonstração Interativa da Correlação em Tempo Contínuo (*Interactive Correlation Explorer*).
- **Exercício I (Extra):** Autocorrelação analítica e computacional do pulso retangular e relação com a Energia ($E_x$).
- **Exercício II (Extra):** Deteção e extração de sinais submersos em ruído forte (Aplicações em Radar e Telecomunicações).


In [ ]:
# Configuração e importação de módulos científicos em Python
import numpy as np
import scipy.signal as signal
import sympy as sp
import matplotlib.pyplot as plt

# Configuração estética global dos gráficos
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.4
plt.rcParams['grid.linestyle'] = '--'
plt.rcParams['figure.figsize'] = (10, 5)

print("Ambiente Python configurado com sucesso para a aula de Sinais e Sistemas!")


---
## 1. Fundamentos Teóricos: Medição de Semelhança entre Sinais

### 1.1 O que é a Correlação?
Antes de abordarmos a convolução em sistemas LIT, a operação mais intuitiva entre dois sinais é a **Correlação**.
A correlação responde à questão de engenharia:  
> *"Quão semelhante é o sinal $x(t)$ com uma versão atrasada de $y(t)$ por um desfasamento temporal $\tau$?"*

Na correlação, **não se efetua a inversão temporal** do sinal deslocado (*Slide & Integrate*):

- **Correlação Cruzada (*Cross-correlation*):**
  - **Tempo Contínuo:**
    $$r_{xy}(\tau) = \int_{-\infty}^{+\infty} x(t + \tau) y^*(t) dt = \int_{-\infty}^{+\infty} x(t) y^*(t - \tau) dt$$
  - **Tempo Discreto:**
    $$r_{xy}[m] = \sum_{n=-\infty}^{+\infty} x[n + m] y^*[n] = \sum_{n=-\infty}^{+\infty} x[n] y^*[n - m]$$

- **Autocorrelação (*Autocorrelation*):** É a correlação de um sinal consigo mesmo desfasado no tempo:
  - **Tempo Contínuo (Sinais de Energia):**
    $$R_{xx}(\tau) = \int_{-\infty}^{+\infty} x(t) x^*(t - \tau) dt$$
  - **Tempo Discreto (Sinais de Energia):**
    $$R_{xx}[m] = \sum_{n=-\infty}^{+\infty} x[n] x^*[n - m]$$

---

### 1.2 Propriedades Notáveis da Autocorrelação:
1. **Simetria Par:** Para sinais reais ($x(t) \in \mathbb{R}$), a autocorrelação é uma função **par**:
   $$R_{xx}(-\tau) = R_{xx}(\tau) \qquad \text{e} \qquad R_{xx}[-m] = R_{xx}[m]$$
2. **Máximo Global na Origem (Energia Total):**
   $$R_{xx}(0) = \int_{-\infty}^{+\infty} |x(t)|^2 dt = E_x \qquad \text{e} \qquad |R_{xx}(\tau)| \leq R_{xx}(0), \; \forall \tau$$
   *(O sinal atinge o grau máximo de semelhança consigo próprio quando o atraso é nulo, $\tau=0$).*
3. **Preservação de Periodicidade (Sinais de Potência):**
   Se $x(t)$ é periódico com período $T_0$, $R_{xx}(\tau)$ é **estritamente periódico** com o mesmo período fundamental $T_0$, e o valor na origem é a potência média: $R_{xx}(0) = P_x$.
4. **Ligação Fundamental com a Convolução:**
   $$R_{xx}(\tau) = x(\tau) * x^*(-\tau) \qquad \text{e} \qquad R_{xx}[m] = x[m] * x^*[-m]$$


---
## 2. Demonstração Interativa da Correlação Contínua (*Interactive Correlation Explorer*)

Para consolidar a intuição física da correlação (*Slide & Integrate*), explore o módulo interativo abaixo.
Arraste o **slider do desfasamento temporal $\tau$** e observe:
1. O sinal de referência fixo $x(t)$ (em azul) e o sinal deslocado $y(t - \tau)$ (em verde — **sem inversão**).
2. A área do produto pontual $x(t) \cdot y(t - \tau)$ sombreada em amarelo.
3. O traçado da função de correlação cruzada $r_{xy}(\tau)$ no painel inferior, com o pico máximo no ponto de maior alinhamento.


In [ ]:
# Módulo Interativo de Correlação Contínua (ipywidgets + matplotlib)
import numpy as np
import matplotlib.pyplot as plt

def plot_correlacao_frame(tau_val=0.0):
    t_axis = np.linspace(-4.0, 4.0, 1500)
    dt = t_axis[1] - t_axis[0]
    
    # 1. Definição de dois pulsos para correlação cruzada
    x_t = np.maximum(0.0, 1.0 - np.abs(t_axis))
    y_t_shifted = np.where(np.abs(t_axis - tau_val) <= 0.75, 1.0, 0.0)
    
    # Produto pontual x(t) * y(t - tau)
    produto = x_t * y_t_shifted
    r_val = float(np.sum(produto) * dt)
    
    # 2. Curva completa de correlação
    tau_full = np.linspace(-3.5, 3.5, 400)
    r_full = []
    for tau_i in tau_full:
        y_i = np.where(np.abs(t_axis - tau_i) <= 0.75, 1.0, 0.0)
        r_full.append(float(np.sum(x_t * y_i) * dt))
    r_full = np.array(r_full)
    
    # 3. Construção dos Gráficos
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7.5))
    
    # Subplot 1: Domínio temporal t
    ax1.plot(t_axis, x_t, 'b-', lw=2.5, label=r'$x(t)$ Fixo (Triangular)')
    ax1.plot(t_axis, y_t_shifted, 'g--', lw=2.0, label=f'$y(t - ({tau_val:.2f}))$ Deslocado')
    ax1.fill_between(t_axis, produto, color='gold', alpha=0.55, 
                     label=f'Área de Coerência / Produto (Integral = {r_val:.3f})')
    ax1.axvline(tau_val, color='darkgreen', linestyle=':', label=f'Atraso tau = {tau_val:.2f} s')
    ax1.axhline(0, color='black', lw=0.8)
    ax1.set_xlim(-3.5, 3.5)
    ax1.set_ylim(-0.2, 1.3)
    ax1.set_xlabel('Tempo $t$ [s]')
    ax1.set_ylabel('Amplitude')
    ax1.set_title(f'Processo de Correlação: Alinhamento para $\\tau = {tau_val:.2f}$ s (Sem Inversão)', fontweight='bold')
    ax1.legend(loc='upper right', framealpha=0.9)
    
    # Subplot 2: Função de Correlação Cruzada r_xy(tau)
    ax2.plot(tau_full, r_full, 'm-', lw=2.5, label=r'Correlação Cruzada $r_{xy}(\tau)$')
    
    tau_past = tau_full[tau_full <= tau_val]
    if len(tau_past) > 0:
        r_past = r_full[:len(tau_past)]
        ax2.plot(tau_past, r_past, 'purple', lw=3.5, label='Trajetória percorrida')
        
    ax2.plot(tau_val, r_val, 'ro', markersize=9, label=f'$r_{{xy}}({tau_val:.2f}) = {r_val:.3f}$')
    ax2.axvline(tau_val, color='gray', linestyle=':')
    ax2.axhline(0, color='black', lw=0.8)
    ax2.set_xlim(-3.5, 3.5)
    ax2.set_ylim(-0.1, max(r_full) * 1.25)
    ax2.set_xlabel(r'Desfasamento Temporal $\tau$ [s]')
    ax2.set_ylabel(r'$r_{xy}(\tau)$')
    ax2.set_title(r'Função de Correlação Resultante $r_{xy}(\tau) = \int_{-\infty}^{\infty} x(t) y(t-\tau) dt$', fontweight='bold')
    ax2.legend(loc='upper right', framealpha=0.9)
    
    plt.tight_layout()
    plt.show()

# Inicialização com ipywidgets
try:
    from ipywidgets import interact, FloatSlider
    import ipywidgets as widgets
    slider_corr = FloatSlider(min=-3.0, max=3.0, step=0.05, value=0.0, 
                              description='Atraso tau:', continuous_update=True,
                              layout=widgets.Layout(width='65%'))
    interact(plot_correlacao_frame, tau_val=slider_corr);
except ImportError:
    print("Módulo 'ipywidgets' não detetado. A renderizar visualização estática de referência:")
    plot_correlacao_frame(tau_val=0.0)


---
## 3. Exercício I (Extra) — Autocorrelação do Pulso Retangular (Tempo Contínuo)

### Enunciado:
Considere o pulso retangular centrado de duração $T$ e amplitude $A$:
$$x(t) = A \cdot \text{rect}\left(\frac{t}{T}\right) = \begin{cases} A, & -\frac{T}{2} \le t \le \frac{T}{2} \\[4pt] 0, & \text{caso contrário} \end{cases}$$

- **a)** Calcule analiticamente a função de autocorrelação $R_{xx}(\tau) = \int_{-\infty}^{+\infty} x(t) x(t - \tau) dt$.
- **b)** Esboce $R_{xx}(\tau)$ e comprove a relação de pico na origem $R_{xx}(0) = E_x$.

---

### Resolução Analítica Passo a Passo:
1. **Suporte Temporal dos Pulsos:**
   - Pulso fixo $x(t)$: $t \in [-T/2, \, T/2]$
   - Pulso deslocado $x(t - \tau)$: $t \in [\tau - T/2, \, \tau + T/2]$
2. **Integração por Casos de Desfasamento $\tau$:**
   - **Caso 1: $\tau < -T$ ou $\tau > T$** $\implies$ Sem sobreposição $\implies \mathbf{R_{xx}(\tau) = 0}$.
   - **Caso 2: $-T \le \tau < 0$** (Deslocamento à esquerda):
     $$R_{xx}(\tau) = \int_{-T/2}^{\tau + T/2} A^2 dt = A^2 \left[ (\tau + T/2) - (-T/2) \right] = \mathbf{A^2(T + \tau)}$$
   - **Caso 3: $0 \le \tau \le T$** (Deslocamento à direita):
     $$R_{xx}(\tau) = \int_{\tau - T/2}^{T/2} A^2 dt = A^2 \left[ T/2 - (\tau - T/2) \right] = \mathbf{A^2(T - \tau)}$$
3. **Expressão Analítica Compacta:**
   $$\mathbf{R_{xx}(\tau) = A^2 (T - |\tau|) \cdot \text{rect}\left(\frac{\tau}{2T}\right) = A^2 T \cdot \Lambda\left(\frac{\tau}{T}\right)}$$
   *(A autocorrelação de um pulso retangular de largura $T$ é uma função **triangular** $\Lambda(\tau/T)$ com base $2T$ e altura $A^2 T$).*
4. **Verificação da Energia:**
   $$R_{xx}(0) = A^2 T = \int_{-\infty}^{+\infty} |x(t)|^2 dt = E_x \quad \checkmark$$


In [ ]:
# Resolução Computacional do Exercício I (Extra)
import numpy as np
import scipy.signal as signal
import matplotlib.pyplot as plt

A = 2.0      # Amplitude do pulso
T = 1.5      # Duração do pulso [s]

t_grid = np.linspace(-3.0, 3.0, 3000)
dt = t_grid[1] - t_grid[0]

# Sinal retangular x(t)
x_rect = np.where(np.abs(t_grid) <= T/2, A, 0.0)

# Solução Teórica: Pulso Triangular
tau_axis = np.linspace(-3.0, 3.0, 3000)
R_teorica = np.where(np.abs(tau_axis) <= T, (A**2) * (T - np.abs(tau_axis)), 0.0)

# Solução Numérica via correlate
R_numerica = signal.correlate(x_rect, x_rect, mode='same') * dt

# Visualização
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

ax1.plot(t_grid, x_rect, 'b-', lw=2.5, label=f'Pulso Retangular $x(t)$ ($A={A}$, $T={T}$s)')
ax1.axhline(0, color='black', lw=0.8)
ax1.set_ylabel('Amplitude $x(t)$')
ax1.set_title('Sinal de Entrada: Pulso Retangular Centrado', fontweight='bold')
ax1.legend(loc='upper right')

ax2.plot(tau_axis, R_teorica, 'r-', lw=2.5, label=r'Solução Analítica: $R_{xx}(\tau) = A^2 (T - |\tau|)$')
ax2.plot(t_grid, R_numerica, 'k--', lw=1.5, label='Convolução Numérica (scipy.signal.correlate)')
ax2.axvline(0, color='gray', linestyle=':')
ax2.axhline(0, color='black', lw=0.8)

# Destaque para a Energia R_xx(0)
E_x = (A**2) * T
ax2.plot(0, E_x, 'ro', markersize=8)
label_energia = f'Energia $E_x = R_{{xx}}(0) = {E_x:.2f}$ J'
ax2.annotate(label_energia, xy=(0, E_x), xytext=(0.3, E_x - 0.7),
             arrowprops=dict(facecolor='black', shrink=0.08, width=1, headwidth=6),
             fontweight='bold', color='darkred')

ax2.set_xlabel(r'Desfasamento Temporal $\tau$ [s]')
ax2.set_ylabel(r'$R_{xx}(\tau)$')
ax2.set_title('Função de Autocorrelação $R_{xx}(\tau)$: Triângulo de base $2T$', fontweight='bold')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()

print(f"Energia Teórica do Pulso: E_x = {E_x:.4f} J")
print(f"Pico Numérico Obtido na Origem: R_xx(0) = {np.max(R_numerica):.4f} J")


---
## 4. Exercício II (Extra) — Deteção de Sinais Submersos em Ruído Forte (RADAR / Comunicações)

### Enunciado:
Considere um sinal discreto sinusoidal puro $s[n] = A \cos(\omega_0 n)$ corrompido por ruído branco gaussiano aditivo $w[n]$ de média nula e variância $\sigma_w^2$:
$$x[n] = s[n] + w[n]$$
com relação sinal-ruído negativa ($\text{SNR} = -6\text{ dB}$, ou seja, a potência do ruído é 4 vezes superior à do sinal).

- **a)** Deduza analiticamente a autocorrelação teórica $R_{xx}[m] = R_{ss}[m] + R_{ww}[m]$.
- **b)** Mostre como a autocorrelação isola o ruído na origem ($m=0$) e recupera a periodicidade $N_0 = 2\pi/\omega_0$ para $m \neq 0$.
- **c)** Simule computacionalmente a extração do sinal e estimativa do período fundamental.

---

### Resolução Analítica:
1. **Autocorrelação da Sinusoide Pura:**
   $$R_{ss}[m] = \lim_{N \to \infty} \frac{1}{2N+1} \sum_{n=-N}^{N} A \cos(\omega_0 n) A \cos(\omega_0(n-m)) = \mathbf{\frac{A^2}{2} \cos(\omega_0 m)}$$
2. **Autocorrelação do Ruído Branco:**
   Como o ruído branco é completamente descorrelacionado para desfasamentos $m \neq 0$:
   $$R_{ww}[m] = \sigma_w^2 \cdot \delta[m]$$
3. **Autocorrelação Total do Sinal Ruidoso:**
   Como o sinal e o ruído são mutuamente descorrelacionados ($R_{sw}[m] = 0$):
   $$\mathbf{R_{xx}[m] = \frac{A^2}{2} \cos(\omega_0 m) + \sigma_w^2 \delta[m]}$$
4. **Conclusão de Engenharia:**
   - Para $m = 0$: $R_{xx}[0] = \frac{A^2}{2} + \sigma_w^2 = P_{\text{sinal}} + P_{\text{ruído}}$ (Potência Total).
   - Para $m \neq 0$: o termo do ruído desaparece ($\sigma_w^2 \delta[m] = 0$), restando apenas a sinusoide periódica:
     $$R_{xx}[m] = \frac{A^2}{2} \cos(\omega_0 m)$$
   A autocorrelação **elimina o ruído**, permitindo estimar frequências e sincronizar recetores mesmo sob ruído severo.


In [ ]:
# Resolução Computacional do Exercício II (Extra)
import numpy as np
import matplotlib.pyplot as plt

# 1. Parâmetros de Simulação
np.random.seed(42)
N = 600
n = np.arange(N)
f0 = 0.05           # Frequência normalizada (Período N0 = 1/f0 = 20 amostras)
omega0 = 2 * np.pi * f0
A = 1.0             # Potência do sinal Ps = A^2/2 = 0.5

# 2. Sinal Útil e Ruído Branco (SNR = -6 dB => Pw = 1.0)
s_n = A * np.cos(omega0 * n)
w_n = np.random.normal(0, 1.0, N)
x_n = s_n + w_n

# 3. Autocorrelação Amostral
R_xx = np.correlate(x_n, x_n, mode='full') / N
lags = np.arange(-(N - 1), N)

mask_lags = (lags >= -60) & (lags <= 60)
lags_vis = lags[mask_lags]
R_vis = R_xx[mask_lags]

# 4. Visualização
fig, axs = plt.subplots(3, 1, figsize=(11, 9))

# Sinal Puro
axs[0].plot(n[:120], s_n[:120], 'g-', lw=2, label=r'Sinal Puro $s[n] = \cos(2\pi \cdot 0.05 n)$ ($N_0 = 20$ amostras)')
axs[0].set_title('Exercício II (Extra): Deteção e Extração de Sinais Submersos em Ruído Forte', fontweight='bold')
axs[0].set_ylabel('Amplitude $s[n]$')
axs[0].set_ylim(-1.8, 1.8)
axs[0].legend(loc='upper right')

# Sinal Ruidoso Recebido (SNR = -6 dB)
axs[1].plot(n[:120], x_n[:120], 'k-', lw=1.2, alpha=0.85, label='Sinal Recebido $x[n] = s[n] + w[n]$ (SNR = -6 dB)')
axs[1].set_ylabel('Amplitude $x[n]$')
axs[1].set_ylim(-3.5, 3.5)
axs[1].legend(loc='upper right')

# Autocorrelação R_xx[m]
axs[2].plot(lags_vis, R_vis, 'r-', lw=2.2, label=r'Autocorrelação $R_{xx}[m]$')
axs[2].axvline(0, color='gray', linestyle=':')
for peak_lag in [-40, -20, 20, 40]:
    axs[2].axvline(peak_lag, color='blue', linestyle='--', alpha=0.6)
    axs[2].plot(peak_lag, R_vis[lags_vis == peak_lag][0], 'bo')

axs[2].set_xlabel('Desfasamento Discreto $m$ (Lags)')
axs[2].set_ylabel(r'$R_{xx}[m]$')
axs[2].set_title(r'Autocorrelação $R_{xx}[m]$: Ruído isolado em $m=0$; Periodicidade $N_0=20$ recuperada!', fontweight='bold')
axs[2].legend(loc='upper right')

plt.tight_layout()
plt.show()

print("Período fundamental identificado via picos de autocorrelação: N0 = 20 amostras.")


---
## 5. Síntese e Conclusões da Parte 1

| Conceito | Fórmula em Tempo Contínuo | Fórmula em Tempo Discreto | Propriedade Chave |
| :--- | :--- | :--- | :--- |
| **Correlação Cruzada** | $r_{xy}(\tau) = \int_{-\infty}^{\infty} x(t) y^*(t-\tau) dt$ | $r_{xy}[m] = \sum x[n] y^*[n-m]$ | Alinhamento sem inversão temporal (*Slide & Integrate*). |
| **Autocorrelação** | $R_{xx}(\tau) = \int_{-\infty}^{\infty} x(t) x^*(t-\tau) dt$ | $R_{xx}[m] = \sum x[n] x^*[n-m]$ | Função par: $R_{xx}(-\tau) = R_{xx}(\tau)$; Pico na origem: $R_{xx}(0)=E_x$. |
| **Deteção em Ruído** | $R_{xx}(\tau) = R_{ss}(\tau) + R_{ww}(\tau)$ | $R_{xx}[m] = R_{ss}[m] + \sigma_w^2 \delta[m]$ | O ruído fica confinado à origem; a periodicidade do sinal útil é preservada. |
